# Dataset Curation
This process included us combining two different datasets, one from UNDP HDI and other from WBG API. 
This file's purpose is just curation, not preprocessing, cleaning or imputing data.

### Following things are still yet to be done (just for curation):
- Merging the two datasets silently erased multiple country codes that only existed in one df but not the other,
we need to somehow make that NOT happen (the project doc explicilty tells us to be careful with this)

- Making this into a proper .py file instead of a .ipynb file

In [73]:
import pandas as pd
import numpy as np
import wbgapi as wb

# Required indicator list as per project doc
codes = ['NY.GDP.MKTP.KD.ZG','FP.CPI.TOTL.ZG','SL.UEM.TOTL.ZS','SP.DYN.IMRT.IN',
         'EG.ELC.ACCS.ZS','IT.NET.USER.ZS','NV.AGR.TOTL.ZS','NV.IND.TOTL.ZS','NE.TRD.GNFS.ZS']

In [74]:
# Converting series into df with proper column names for merging
df_1 = wb.data.DataFrame(codes, time=range(2018, 2024), labels=False)
indicators = df_1['YR2023'][:]
idx = indicators.index.levels
c = len(idx[1])
indicators = pd.DataFrame(indicators.values.reshape(-1,c),index=idx[0].values, columns=idx[1].values).reset_index()

indicators['iso3'] = indicators['index']
indicators = indicators.drop(columns=['index'])

In [75]:
# Merging Datasets by country
df_2 = pd.read_csv("data/HDR25_Composite_indices_complete_time_series.csv", encoding='latin1')
predictor = df_2[['iso3','hdi_2023', 'region']][:]
dataset = pd.merge(predictor, indicators, on='iso3')

# These 2 countries do not have a registered HDI_2023 score
df_2.iloc[115] # MCO
df_2.iloc[90] # PRK

iso3                                              PRK
country           Korea (Democratic People's Rep. of)
hdicode                Other Countries or Territories
region                                            EAP
hdi_rank_2023                                     NaN
                                 ...                 
pop_total_2019                              26.037633
pop_total_2020                              26.136312
pop_total_2021                              26.232533
pop_total_2022                              26.328845
pop_total_2023                              26.418204
Name: 90, Length: 1112, dtype: object

In [76]:
# Encoding our target response variable as per project requirement
dataset['Level'] = np.where(dataset['hdi_2023'] > 0.800, 2, np.where(dataset['hdi_2023'] < 0.550, 0, 1))

In [77]:
# Renaming columns for better readability
dataset.rename(columns = {
    'NV.AGR.TOTL.ZS' : 'Agriculture',
    'NV.IND.TOTL.ZS' : 'Industry',
    'NY.GDP.MKTP.KD.ZG' : 'GDP_Growth',
    'FP.CPI.TOTL.ZG': 'Inflation_consumer_prices',
    'SL.UEM.TOTL.ZS': 'Unemployment_total',
    'SP.DYN.IMRT.IN': 'Infant_mortality',
    'EG.ELC.ACCS.ZS': 'Access_to_electricity',
    'IT.NET.USER.ZS':'Individuals_using_the_Internet',
    'NE.TRD.GNFS.ZS':'Trade',
    'iso3': 'Country',
    'region': 'Region'

}, inplace=True)

# Creating feature as per project doc
dataset['Services'] = 100 - dataset['Agriculture'] - dataset['Industry']

dataset['Dominant_Sector'] = np.where((dataset['Services'] > dataset['Agriculture']), 
        np.where(dataset['Services'] > dataset['Industry'], 'Services', 'Industry'),
        np.where(dataset['Agriculture'] > dataset['Industry'],'Agriculture', 'Industry')
        )
dataset.loc[dataset['Services'].isna(), 'Dominant_Sector'] = pd.NA

dataset.loc[[90, 115], "Level"] = pd.NA
dataset.drop(columns = ['hdi_2023'], inplace=True)

In [78]:
# Ordering columns
cols = ['Country', 'Region', 'Level', 'Agriculture', 'Industry', 'Services', 'Dominant_Sector', 'Access_to_electricity', 'Inflation_consumer_prices', 'Individuals_using_the_Internet', 'Trade', 'GDP_Growth', 'Unemployment_total', 'Infant_mortality' ]
dataset = dataset[cols]

In [79]:
dataset

,Country,Region,Level,Agriculture,Industry,Services,Dominant_Sector,Access_to_electricity,Inflation_consumer_prices,Individuals_using_the_Internet,Trade,GDP_Growth,Unemployment_total,Infant_mortality
0,AFG,SA,0.0,34.743247,13.449823,51.806930,Services,85.3,-4.644709,15.928400,66.999578,2.266944,14.008,49.9
1,ALB,ECA,2.0,16.274239,23.160712,60.565049,Services,100.0,4.758346,83.135592,82.352493,4.015417,10.669,8.3
2,DZA,AS,1.0,13.369245,37.547655,49.083100,Services,100.0,9.322174,76.197601,43.739798,4.100000,11.645,19.7
3,AND,NaN,2.0,0.478492,12.428226,87.093282,Services,100.0,NaN,94.214996,NaN,2.583555,NaN,2.5
4,AGO,SSA,1.0,20.909337,35.701247,43.389416,Services,51.1,13.644102,37.885899,53.607013,1.320166,14.051,33.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,VEN,LAC,1.0,NaN,NaN,NaN,NaN,100.0,NaN,76.331902,28.094432,4.022989,5.407,21.2
191,VNM,EAP,1.0,11.871327,37.593288,50.535385,Services,99.8,3.252893,78.080000,164.995585,4.978660,1.645,12.4
192,YEM,AS,0.0,NaN,NaN,NaN,NaN,83.6,NaN,NaN,NaN,NaN,16.944,34.9
193,ZMB,SSA,1.0,2.230120,35.107778,62.662102,Services,51.1,10.884532,16.450701,78.241887,5.367943,5.861,35.0
